# Спринт ревью №2
## Тема: «Наследование, абстрактный класс и интерфейсы»

**Язык:** C# (Polyglot Notebook / .NET Interactive)

**Цель работы:** создать классы, представляющие структуру коммерческого предприятия, используя простое, сложное (многоуровневое) и множественное наследование (через интерфейсы).

**Иерархия классов:**

```
Employee  →  Manager  →  ProjectManager  (+ IReportable, ITeamLeader)
```

> Ячейки нужно запускать строго сверху вниз.

## Задание 1. Базовый класс `Employee`

Класс содержит поля (свойства) `Name`, `Position`, `Salary` (тип `double`) и метод `PrintInfo()`.
Метод помечен как `virtual`, чтобы производные классы могли его переопределить (`override`).

In [1]:
public class Employee
{
    public string Name { get; set; }       // имя сотрудника
    public string Position { get; set; }   // должность
    public double Salary { get; set; }     // зарплата

    public Employee(string name, string position, double salary)
    {
        Name = name;
        Position = position;
        Salary = salary;
    }

    // virtual - разрешает переопределение в наследниках
    public virtual void PrintInfo()
    {
        Console.WriteLine($"Имя: {Name}");
        Console.WriteLine($"Должность: {Position}");
        Console.WriteLine($"Зарплата: {Salary:F2}");
    }
}

The below script needs to be able to find the current output cell; this is an easy method to get it.

## Задание 2. Простое наследование: класс `Manager`

`Manager` наследует `Employee`, добавляет свойство `Department` и переопределяет `PrintInfo()`.
Через `base.PrintInfo()` вызывается реализация базового класса, затем выводится отдел.

In [2]:
public class Manager : Employee
{
    public string Department { get; set; }  // отдел, которым управляет менеджер

    public Manager(string name, string position, double salary, string department)
        : base(name, position, salary)
    {
        Department = department;
    }

    public override void PrintInfo()
    {
        base.PrintInfo();
        Console.WriteLine($"Отдел: {Department}");
    }
}

## Задание 4 (часть 1). Интерфейсы `IReportable` и `ITeamLeader`

В C# класс может наследоваться только от одного класса, но реализовывать сколько угодно интерфейсов - так достигается множественное наследование.

> Интерфейсы объявляются **до** `ProjectManager`, потому что класс должен знать о них в момент объявления (иначе в ноутбуке возникнет ошибка компиляции).

In [3]:
public interface IReportable
{
    string GenerateReport();   // возвращает строку с отчётом
}

public interface ITeamLeader
{
    string[] GetTeamMembers(); // возвращает список членов команды
}

## Задание 3 и 4 (часть 2). Класс `ProjectManager`

`ProjectManager` наследует `Manager` (многоуровневое наследование: Employee → Manager → ProjectManager), добавляет свойство `ProjectName`, переопределяет `PrintInfo()` и реализует оба интерфейса.

In [4]:
public class ProjectManager : Manager, IReportable, ITeamLeader
{
    public string ProjectName { get; set; }  // название проекта

    private readonly List<string> _teamMembers = new List<string>();

    public ProjectManager(string name, string position, double salary,
                          string department, string projectName)
        : base(name, position, salary, department)
    {
        ProjectName = projectName;
    }

    public void AddTeamMember(string member)
    {
        _teamMembers.Add(member);
    }

    public override void PrintInfo()
    {
        base.PrintInfo();   // Name, Position, Salary, Department
        Console.WriteLine($"Проект: {ProjectName}");
    }

    // Реализация IReportable
    public string GenerateReport()
    {
        return $"Отчёт: проектный менеджер {Name} ({Position}) руководит проектом " +
               $"\"{ProjectName}\" в отделе \"{Department}\". " +
               $"Количество участников команды: {_teamMembers.Count}.";
    }

    // Реализация ITeamLeader
    public string[] GetTeamMembers()
    {
        return _teamMembers.ToArray();
    }
}

## Задание 5. Тестирование

Создаём по одному экземпляру каждого класса, вызываем `PrintInfo()` и проверяем методы интерфейсов.

In [6]:
var employee1 = new Employee("Иван Петров", "Бухгалтер", 55000);
var manager1 = new Manager("Анна Смирнова", "Менеджер по продажам", 85000, "Отдел продаж");
var projectManager1 = new ProjectManager("Дмитрий Орлов", "Проектный менеджер", 120000,
                                         "IT-отдел", "Разработка CRM-системы");

projectManager1.AddTeamMember("Алексей Козлов");
projectManager1.AddTeamMember("Мария Иванова");
projectManager1.AddTeamMember("Сергей Волков");

Console.WriteLine("Employee");
employee1.PrintInfo();

Console.WriteLine();
Console.WriteLine("Manager");
manager1.PrintInfo();

Console.WriteLine();
Console.WriteLine("ProjectManager");
projectManager1.PrintInfo();

Console.WriteLine();
Console.WriteLine("IReportable.GenerateReport()");
Console.WriteLine(projectManager1.GenerateReport());

Console.WriteLine();
Console.WriteLine("ITeamLeader.GetTeamMembers()");
foreach (var member in projectManager1.GetTeamMembers())
{
    Console.WriteLine($"- {member}");
}

Employee
Имя: Иван Петров
Должность: Бухгалтер
Зарплата: 55000,00

Manager
Имя: Анна Смирнова
Должность: Менеджер по продажам
Зарплата: 85000,00
Отдел: Отдел продаж

ProjectManager
Имя: Дмитрий Орлов
Должность: Проектный менеджер
Зарплата: 120000,00
Отдел: IT-отдел
Проект: Разработка CRM-системы

IReportable.GenerateReport()
Отчёт: проектный менеджер Дмитрий Орлов (Проектный менеджер) руководит проектом "Разработка CRM-системы" в отделе "IT-отдел". Количество участников команды: 3.

ITeamLeader.GetTeamMembers()
- Алексей Козлов
- Мария Иванова
- Сергей Волков


### Демонстрация полиморфизма и работы через интерфейсы

Один и тот же вызов `PrintInfo()` для переменной типа `Employee` выполняет версию метода того класса, объектом которого она является.

In [10]:
var staffList = new List<Employee> { employee1, manager1, projectManager1 };

foreach (var person in staffList)
{
    Console.WriteLine($"{person.GetType().Name}");
    person.PrintInfo();
    Console.WriteLine();
}

// Работа через интерфейсы
IReportable reportable = projectManager1;
ITeamLeader leader = projectManager1;

Console.WriteLine(reportable.GenerateReport());
Console.WriteLine("Количество участников: " + leader.GetTeamMembers().Length);

Console.WriteLine();
Console.WriteLine($"projectManager1 is Employee: {projectManager1 is Employee}");
Console.WriteLine($"projectManager1 is Manager: {projectManager1 is Manager}");
Console.WriteLine($"projectManager1 is IReportable: {projectManager1 is IReportable}");
Console.WriteLine($"projectManager1 is ITeamLeader: {projectManager1 is ITeamLeader}");

Employee
Имя: Иван Петров
Должность: Бухгалтер
Зарплата: 55000,00

Manager
Имя: Анна Смирнова
Должность: Менеджер по продажам
Зарплата: 85000,00
Отдел: Отдел продаж

ProjectManager
Имя: Дмитрий Орлов
Должность: Проектный менеджер
Зарплата: 120000,00
Отдел: IT-отдел
Проект: Разработка CRM-системы

Отчёт: проектный менеджер Дмитрий Орлов (Проектный менеджер) руководит проектом "Разработка CRM-системы" в отделе "IT-отдел". Количество участников команды: 3.
Количество участников: 3

projectManager1 is Employee: True
projectManager1 is Manager: True
projectManager1 is IReportable: True
projectManager1 is ITeamLeader: True


## Вывод

В ходе лабораторной работы были реализованы:

1. **Базовый класс `Employee`** с полями `Name`, `Position`, `Salary` и виртуальным методом `PrintInfo()`.
2. **Простое наследование:** `Manager` наследует `Employee`, добавляет `Department` и переопределяет `PrintInfo()` (`override`), используя `base.PrintInfo()`.
3. **Многоуровневое наследование:** `ProjectManager` наследует `Manager`, добавляет `ProjectName` и дополняет вывод `PrintInfo()`.
4. **Множественное наследование через интерфейсы:** т.к. C# не поддерживает множественное наследование классов, `ProjectManager` реализует интерфейсы `IReportable` (`GenerateReport()`) и `ITeamLeader` (`GetTeamMembers()`).
5. **Тестирование:** созданы объекты всех трёх классов, проверены `PrintInfo()`, `GenerateReport()`, `GetTeamMembers()`, а также полиморфизм и абстрактный класс.

Наследование позволяет повторно использовать код, переопределение методов - менять поведение в потомках, а интерфейсы - задавать контракт, который могут выполнять классы из разных иерархий.